In [96]:
import pandas as pd

df = pd.read_csv(
    "C:/Users/usuario/Documents/Ciencia de Datos/TPFINAL/Base Aprender Estudiantes 6 Grado Primaria 2023 (3).csv",
    sep=";",       # si no funciona con "," probá con ";"
    encoding="latin1"  # si no funciona, probá "utf-8"
)

In [97]:
import numpy as np
import pandas as pd

# Convertir todas las columnas a numérico si es posible
df = df.apply(pd.to_numeric, errors='coerce')

In [98]:
# Reemplazar -9/-8/-6 por NaN y asegurarnos de que sean numéricas
cols = ["sector", "ambito", "ap01"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").replace({-9: np.nan, -8: np.nan, -6: np.nan})

# (Opcional) eliminar filas que tengan NaN en cualquiera de estas 3 columnas
df = df.dropna(subset=cols)
# sector: 1=Estatal, 2=Privado → col 0/1 para "Privado"
df["sector_privado"] = (df["sector"] == 2).astype(int)

# ambito: 1=Rural, 2=Urbano → col 0/1 para "Urbano"
df["ambito_urbano"] = (df["ambito"] == 2).astype(int)

# (Recomendado para PCA) quitar las originales nominales
df = df.drop(columns=["sector", "ambito"])

In [99]:
# Limpiar y crear dummies para ap03
df["ap03"] = pd.to_numeric(df["ap03"], errors="coerce").replace({-9: np.nan, -8: np.nan, -6: np.nan})
df = df.dropna(subset=["ap03"])  # eliminar filas con NaN

df = pd.get_dummies(df, columns=["ap03"], prefix="sexo", drop_first=True)
# crea sexo_2 (Femenino) y sexo_3 (X)
df = df.astype({col: "int" for col in df.select_dtypes(bool).columns})

In [18]:
print(df.head(10))

           ï»¿ID1  jurisdiccion  seccion  idalumno  ap01  ap02  ap04  ap05a  \
0  20004002000400            50        1    247287   2.0   1.0   1.0    1.0   
1  20004002000400            50        2    247266   3.0   9.0   1.0    1.0   
2  20004002000400            50        1    247288   3.0   9.0   1.0    2.0   
3  20004002000400            50        1    247289   2.0   1.0   1.0    1.0   
4  20004002000400            50        1    247290   2.0  12.0   1.0    1.0   
5  20004002000400            50        1    247291   3.0   8.0   1.0    1.0   
6  20004002000400            50        1    247292   2.0  10.0   1.0   -9.0   
7  20004002000400            50        1    247293   2.0   2.0   1.0   -9.0   
8  20004002000400            50        1    247294   3.0   7.0   1.0    1.0   
9  20004002000400            50        1    247295   3.0   8.0   1.0    1.0   

   ap05b  ap05c  ...  migracion  sobreedad  Nivel_Ed_Madre  Nivel_Ed_Padre  \
0    1.0    1.0  ...          2          1          

In [100]:
# Nos aseguramos de que idalumno sea numérico (por si acaso)
df["idalumno"] = pd.to_numeric(df["idalumno"], errors="coerce")

# Agrupar por idalumno (en caso de que haya más de una observación por alumno)
# y tomar la media o el máximo según corresponda
df1 = (
    df.groupby("idalumno", as_index=False)
      .agg({
          "ap01": "mean",            # Edad (promedio si hay duplicados)
          "sector_privado": "max",   # 1 si alguna observación es privada
          "ambito_urbano": "max",    # 1 si alguna observación es urbana
          "sexo_2.0": "max",           # 1 si Femenino
          "sexo_3.0": "max"            # 1 si X
      })
      .rename(columns={"ap01": "ap01_edad_index"})
)

In [101]:
# Definir columnas del bloque
ap05_cols = [f"ap05{c}" for c in list("abcdefgh")]

# Limpiar y convertir valores
df[ap05_cols] = df[ap05_cols].apply(pd.to_numeric, errors="coerce").replace({-9: np.nan, -8: np.nan, -6: np.nan})

# Mapear respuestas: 1 = Sí, 2 = No
df[ap05_cols] = df[ap05_cols].replace({1: 1, 2: 0})

# Crear el índice como promedio de respuestas (NaN no se cuenta)
df["ap05_index"] = df[ap05_cols].mean(axis=1, skipna=True)

# Agrupar por idalumno y unir a df1
ap05_index = df.groupby("idalumno", as_index=False)["ap05_index"].mean()
df1 = df1.merge(ap05_index, on="idalumno", how="left")

C:\Users\usuario\AppData\Local\Temp\ipykernel_20312\501448138.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ap05_index"] = df[ap05_cols].mean(axis=1, skipna=True)


In [102]:
# Definir columnas del bloque
ap06_cols = [f"ap06{c}" for c in list("abcde")]

# Limpiar valores inválidos
df[ap06_cols] = df[ap06_cols].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# Crear índice como promedio de las respuestas válidas
df["ap06_index"] = df[ap06_cols].mean(axis=1, skipna=True)

# Agrupar por idalumno y unir al df1
ap06_index = df.groupby("idalumno", as_index=False)["ap06_index"].mean()
df1 = df1.merge(ap06_index, on="idalumno", how="left")

C:\Users\usuario\AppData\Local\Temp\ipykernel_20312\476429443.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ap06_index"] = df[ap06_cols].mean(axis=1, skipna=True)


In [103]:
# Definir columnas del bloque
ap07_cols = ["ap07a", "ap07b"]

# Limpiar valores inválidos
df[ap07_cols] = df[ap07_cols].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# Crear índice como promedio de las respuestas válidas
df["ap07_index"] = df[ap07_cols].mean(axis=1, skipna=True)

# Agrupar por idalumno y unir al df1
ap07_index = df.groupby("idalumno", as_index=False)["ap07_index"].mean()
df1 = df1.merge(ap07_index, on="idalumno", how="left")

C:\Users\usuario\AppData\Local\Temp\ipykernel_20312\1789343149.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ap07_index"] = df[ap07_cols].mean(axis=1, skipna=True)


In [104]:
print(df1.head(10))

   idalumno  ap01_edad_index  sector_privado  ambito_urbano  sexo_2.0  \
0         1              2.0               0              1         1   
1         2              2.0               0              1         0   
2         3              2.0               0              1         1   
3         4              3.0               0              1         1   
4         5              2.0               0              1         1   
5         6              2.0               0              1         0   
6         7              2.0               0              1         1   
7         8              2.0               0              1         0   
8         9              2.0               0              1         0   
9        10              2.0               0              1         0   

   sexo_3.0  ap05_index  ap06_index  ap07_index  
0         0      1.0000       2.200        2.75  
1         0      0.6875       1.875        1.00  
2         0      0.5625       1.350        1.0

In [105]:
# Definir columnas del bloque
ap09_cols = [f"ap09{c}" for c in list("abcdefghijk")]

# Limpiar valores inválidos
df[ap09_cols] = df[ap09_cols].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# Mapear respuestas: 1 = Sí, 2 = No
df[ap09_cols] = df[ap09_cols].replace({1: 1, 2: 0})

# Crear índice como promedio de respuestas válidas (ignora NaN)
df["ap09_index"] = df[ap09_cols].mean(axis=1, skipna=True)

# Agrupar por idalumno y unir a df1
ap09_index = df.groupby("idalumno", as_index=False)["ap09_index"].mean()
df1 = df1.merge(ap09_index, on="idalumno", how="left")

C:\Users\usuario\AppData\Local\Temp\ipykernel_20312\1902926127.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ap09_index"] = df[ap09_cols].mean(axis=1, skipna=True)


In [106]:
# Variables del bloque
cols_ap10_15 = ["ap10", "ap11", "ap12", "ap13", "ap14", "ap15"]

# Asegurar formato numérico y limpiar códigos inválidos
df[cols_ap10_15] = df[cols_ap10_15].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# Unir al df1 (puede haber NaN)
df_temp = df[["idalumno"] + cols_ap10_15].copy()
df_temp = df_temp.groupby("idalumno", as_index=False).mean()

# Renombrar columnas para claridad
df_temp = df_temp.rename(columns={
    "ap10": "ap10_libros",
    "ap11": "ap11_si_no",
    "ap12": "ap12_si_no",
    "ap13": "ap13_si_no",
    "ap14": "ap14_cantidad",
    "ap15": "ap15_cantidad"
})

# Agregar al df1 (sin eliminar nada todavía)
df1 = df1.merge(df_temp, on="idalumno", how="left")

In [107]:
# Eliminar de df1 las filas que tengan NA en cualquiera de estas columnas
df1 = df1.dropna(subset=[
    "ap10_libros", "ap11_si_no", "ap12_si_no",
    "ap13_si_no", "ap14_cantidad", "ap15_cantidad"
])

In [31]:
df1= df1.dropna()

VER

In [108]:
# Definir columnas del bloque
cols_ap18_19 = [f"ap18{c}" for c in list("abcd")] + [f"ap19{c}" for c in list("abcd")]

# Convertir a numérico y limpiar valores inválidos
df[cols_ap18_19] = df[cols_ap18_19].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# Mapear respuestas: 1 = Sí → 1, 2 = No → 0
df[cols_ap18_19] = df[cols_ap18_19].replace({1: 1, 2: 0})

# Agrupar por idalumno, usando el máximo (1 si alguna vez fue Sí)
ap18_19 = df.groupby("idalumno", as_index=False)[cols_ap18_19].max()

# Unir a df1
df1 = df1.merge(ap18_19, on="idalumno", how="left")

In [109]:
print(df1.head(10))

   idalumno  ap01_edad_index  sector_privado  ambito_urbano  sexo_2.0  \
0         1              2.0               0              1         1   
1         2              2.0               0              1         0   
2         3              2.0               0              1         1   
3         4              3.0               0              1         1   
4         5              2.0               0              1         1   
5         7              2.0               0              1         1   
6         8              2.0               0              1         0   
7         9              2.0               0              1         0   
8        11              2.0               0              1         0   
9        12              2.5               0              1         1   

   sexo_3.0  ap05_index  ap06_index  ap07_index  ap09_index  ...  \
0         0      1.0000       2.200        2.75    0.772727  ...   
1         0      0.6875       1.875        1.00    0.863636 

In [110]:
# Verificar cuántos duplicados hay
print("Cantidad de filas duplicadas:", df["idalumno"].duplicated().sum())

# Eliminar duplicados, quedándote con la primera ocurrencia
df1 = df1.drop_duplicates(subset="idalumno", keep="first")

# Confirmar que no queden duplicados
print("Duplicados después de limpiar:", df1["idalumno"].duplicated().sum())

Cantidad de filas duplicadas: 24
Duplicados después de limpiar: 0


In [42]:
df1= df1.dropna()

In [111]:
print("Filas antes de limpiar:", df1.shape[0])

Filas antes de limpiar: 517057


In [112]:
# Variables del bloque
cols_ap21_22 = ["ap22"]

# Convertir a numérico y limpiar valores inválidos
df[cols_ap21_22] = df[cols_ap21_22].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# Eliminar filas con NaN en estas preguntas
df = df.dropna(subset=cols_ap21_22)

# Crear dummies para cada variable
dummies_ap22 = pd.get_dummies(df["ap22"], prefix="ap22", dtype=int)

# Unir las dummies al DataFrame original
df_dummies = pd.concat([df[["idalumno"]], dummies_ap22], axis=1)

# Agrupar por idalumno (por si hay repetidos)
df_dummies = df_dummies.groupby("idalumno", as_index=False).max()

# Unir al df1
df1 = df1.merge(df_dummies, on="idalumno", how="left")

In [46]:
print(df1.head(10))

   idalumno  ap01_edad_index  sector_privado  ambito_urbano  sexo_2.0  \
0         2              2.0               0              1         0   
1         5              2.0               0              1         1   
2        15              2.5               0              1         0   
3        18              2.5               0              1         1   
4        19              2.0               0              1         1   
5        21              2.0               0              1         0   
6        32              2.0               0              1         0   
7        65              3.0               0              1         1   
8       131              2.0               0              0         1   
9       171              2.0               0              0         0   

   sexo_3.0  ap05_index  ap06_index  ap07_index  ap09_index  ...  ap19c_y  \
0         0    0.687500       1.875        1.00    0.863636  ...      0.0   
1         0    1.000000       1.900       

In [113]:
# Definir columnas del bloque
cols_ap24 = [f"ap24{c}" for c in list("abcdefghijk")]

# Convertir a numérico y limpiar valores inválidos
df[cols_ap24] = df[cols_ap24].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0
})

# Agrupar por idalumno (por si hay duplicados) y tomar el máximo
# (si alguna vez marcó 1 en alguna observación, queda 1)
ap24 = df.groupby("idalumno", as_index=False)[cols_ap24].max()

# Unir al df1
df1 = df1.merge(ap24, on="idalumno", how="left")

In [49]:
df1= df1.dropna()

In [114]:
print("Filas antes de limpiar:", df1.shape[0])

Filas antes de limpiar: 517057


In [115]:
# Definir columnas del bloque
cols_ap25 = [f"ap25{c}" for c in list("abcdefghij")]

# Convertir a numérico y reemplazar -8 y -9 por 0
df[cols_ap25] = df[cols_ap25].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0
})

# Agrupar por idalumno (por si hay duplicados) y tomar el máximo
# (si alguna vez seleccionó 1, queda 1)
ap25 = df.groupby("idalumno", as_index=False)[cols_ap25].max()

# Unir al df1
df1 = df1.merge(ap25, on="idalumno", how="left")

In [52]:
print(df1.head(10))

   idalumno  ap01_edad_index  sector_privado  ambito_urbano  sexo_2.0  \
0         2              2.0               0              1         0   
1         5              2.0               0              1         1   
2        15              2.5               0              1         0   
3        18              2.5               0              1         1   
4        19              2.0               0              1         1   
5        21              2.0               0              1         0   
6        32              2.0               0              1         0   
7       131              2.0               0              0         1   
8       188              2.0               0              0         1   
9       281              2.0               0              0         0   

   sexo_3.0  ap05_index  ap06_index  ap07_index  ap09_index  ...  ap25a  \
0         0    0.687500       1.875        1.00    0.863636  ...    1.0   
1         0    1.000000       1.900        1.5

In [116]:
# Variable
col_ap26 = "ap26"

# Convertir a numérico y limpiar
df[col_ap26] = pd.to_numeric(df[col_ap26], errors="coerce")

# Reemplazar -9, -8, -6 por NaN
df[col_ap26] = df[col_ap26].replace({-9: np.nan, -8: np.nan, -6: np.nan})

# Calcular la moda (valor más frecuente, ignorando NaN)
moda_ap26 = df[col_ap26].mode()[0]
print("La opción más frecuente en ap26 es:", moda_ap26)

# Reemplazar NaN con la moda
df[col_ap26] = df[col_ap26].fillna(moda_ap26)

# Agrupar por idalumno (por si hay duplicados)
ap26 = df.groupby("idalumno", as_index=False)[col_ap26].max()

# Unir al df1
df1 = df1.merge(ap26, on="idalumno", how="left")

La opción más frecuente en ap26 es: 1.0


In [54]:
print(df1.head(10))

   idalumno  ap01_edad_index  sector_privado  ambito_urbano  sexo_2.0  \
0         2              2.0               0              1         0   
1         5              2.0               0              1         1   
2        15              2.5               0              1         0   
3        18              2.5               0              1         1   
4        19              2.0               0              1         1   
5        21              2.0               0              1         0   
6        32              2.0               0              1         0   
7       131              2.0               0              0         1   
8       188              2.0               0              0         1   
9       281              2.0               0              0         0   

   sexo_3.0  ap05_index  ap06_index  ap07_index  ap09_index  ...  ap25b  \
0         0    0.687500       1.875        1.00    0.863636  ...    0.0   
1         0    1.000000       1.900        1.5

In [117]:
# Definir las columnas del bloque
cols_apA34 = [f"apA34{c}" for c in list("abcdefghijklm")]

# Convertir a numérico y reemplazar valores inválidos por 0
df[cols_apA34] = df[cols_apA34].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0, -6: 0, 2: 0, 1: 1
})

# Crear el índice sumando todos los "Sí" (1) por alumno
apA34_index = (
    df.groupby("idalumno", as_index=False)[cols_apA34]
    .sum()
    .assign(indice_apA34=lambda x: x[cols_apA34].sum(axis=1))
    [["idalumno", "indice_apA34"]]
)

# Unir el índice al df1
df1 = df1.merge(apA34_index, on="idalumno", how="left")

In [56]:
print(df1.head(10))

   idalumno  ap01_edad_index  sector_privado  ambito_urbano  sexo_2.0  \
0         2              2.0               0              1         0   
1         5              2.0               0              1         1   
2        15              2.5               0              1         0   
3        18              2.5               0              1         1   
4        19              2.0               0              1         1   
5        21              2.0               0              1         0   
6        32              2.0               0              1         0   
7       131              2.0               0              0         1   
8       188              2.0               0              0         1   
9       281              2.0               0              0         0   

   sexo_3.0  ap05_index  ap06_index  ap07_index  ap09_index  ...  ap25c  \
0         0    0.687500       1.875        1.00    0.863636  ...    1.0   
1         0    1.000000       1.900        1.5

In [118]:
# Definir las columnas del bloque
cols_apA36 = [f"apA36{c}" for c in list("abcde")]

# Convertir a numérico y limpiar valores inválidos
df[cols_apA36] = df[cols_apA36].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0, -6: 0
})

# Reescalar las respuestas según el puntaje positivo (4 = siempre, 1 = nunca)
mapa_puntaje = {1: 4, 2: 3, 3: 2, 4: 1}
df[cols_apA36] = df[cols_apA36].replace(mapa_puntaje)

# Calcular el índice de bienestar (suma total de los puntajes)
apA36_index = (
    df.groupby("idalumno", as_index=False)[cols_apA36]
    .sum()
    .assign(apA36_positivo=lambda x: x[cols_apA36].sum(axis=1))
    [["idalumno", "apA36_positivo"]]
)

# Unir el índice al df1
df1 = df1.merge(apA36_index, on="idalumno", how="left")

In [119]:
# Definir columnas del bloque negativo
cols_apA36_neg = [f"apA36{c}" for c in list("fgh")]

# Convertir a numérico y limpiar valores inválidos
df[cols_apA36_neg] = df[cols_apA36_neg].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0, -6: 0
})

# Mapear puntajes (1 = Siempre → 4 puntos, 4 = Nunca → 1 punto)
mapa_puntaje = {1: 4, 2: 3, 3: 2, 4: 1}
df[cols_apA36_neg] = df[cols_apA36_neg].replace(mapa_puntaje)

# Calcular el índice de malestar (suma total)
apA36_neg_index = (
    df.groupby("idalumno", as_index=False)[cols_apA36_neg]
    .sum()
    .assign(apA36_index_negativo=lambda x: x[cols_apA36_neg].sum(axis=1))
    [["idalumno", "apA36_index_negativo"]]
)

# Unir al df1
df1 = df1.merge(apA36_neg_index, on="idalumno", how="left")
# Definir columnas del bloque negativo
cols_apA36_neg = [f"apA36{c}" for c in list("fgh")]

# Convertir a numérico y limpiar valores inválidos
df[cols_apA36_neg] = df[cols_apA36_neg].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0, -6: 0
})

# Mapear puntajes (1 = Siempre → 4 puntos, 4 = Nunca → 1 punto)
mapa_puntaje = {1: 4, 2: 3, 3: 2, 4: 1}
df[cols_apA36_neg] = df[cols_apA36_neg].replace(mapa_puntaje)

# Calcular el índice de malestar (suma total)
apA36_neg_index = (
    df.groupby("idalumno", as_index=False)[cols_apA36_neg]
    .sum()
    .assign(apA36_index_negativo=lambda x: x[cols_apA36_neg].sum(axis=1))
    [["idalumno", "apA36_index_negativo"]]
)

# Unir al df1
df1 = df1.merge(apA36_neg_index, on="idalumno", how="left")

VER

In [60]:
# Definir las columnas del bloque
cols_apA38 = [f"apA38{c}" for c in list("abcdefghijk")]

# Convertir a numérico y reemplazar -9 y -8 por 0
df[cols_apA38] = df[cols_apA38].apply(pd.to_numeric, errors="coerce").replace({
    -9: 0, -8: 0
})

# Agrupar por alumno (por si hay duplicados) y tomar el máximo (si alguna vez seleccionó, queda 1)
apA38 = df.groupby("idalumno", as_index=False)[cols_apA38].max()

# Crear el índice sumando todos los motivos seleccionados
apA38["indice_discriminacion"] = apA38[cols_apA38].sum(axis=1)

# Unir al df1
df1 = df1.merge(apA38[["idalumno", "indice_discriminacion"]], on="idalumno", how="left")

C:\Users\usuario\AppData\Local\Temp\ipykernel_20312\918089065.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[cols_apA38] = df[cols_apA38].apply(pd.to_numeric, errors="coerce").replace({


In [62]:
df1["indice_discriminacion"].value_counts().sort_index()

indice_discriminacion
0.0     29738
1.0      4717
2.0      1283
3.0       846
4.0       535
5.0       438
6.0       398
7.0       230
8.0       138
9.0       105
10.0       67
11.0      296
Name: count, dtype: int64

In [71]:
print("Filas antes de limpiar:", df1.shape[0])

Filas antes de limpiar: 38796


In [120]:
# Aseguramos que sean numéricas y válidas
cols_desemp = ["ldesemp", "mdesemp"]
df[cols_desemp] = df[cols_desemp].apply(pd.to_numeric, errors="coerce").replace({
    -9: np.nan, -8: np.nan, -6: np.nan
})

# (Opcional) eliminamos filas con NaN en estas columnas
df = df.dropna(subset=cols_desemp)

# Unimos al df1 por idalumno
df1 = df1.merge(df[["idalumno"] + cols_desemp], on="idalumno", how="left")

In [121]:

# Agregamos NSE_nivel a df1
df1 = df1.merge(df[["idalumno", "NSE_nivel"]], on="idalumno", how="left")


In [122]:
# Limpiar y recodificar variable 'migracion'
df["migracion"] = pd.to_numeric(df["migracion"], errors="coerce").replace({
    -9: np.nan,   # valores faltantes
    2: 0          # 2 -> no migrante
    # 1 queda como migrante (1)
})

# Unir a df1
df1 = df1.merge(df[["idalumno", "migracion"]], on="idalumno", how="left")



In [123]:
# Limpiar y recodificar 'sobreedad'
df["sobreedad"] = pd.to_numeric(df["sobreedad"], errors="coerce").replace({
    -9: np.nan,  # blanco -> NA
    0: 0,        # menor o edad teórica -> 0
    1: 0,        # idem
    2: 1,        # 1 año de sobreedad
    3: 2,        # 2 años
    4: 3         # 3 años o más
})

# Unir al df1
df1 = df1.merge(df[["idalumno", "sobreedad"]], on="idalumno", how="left")


In [124]:
# Asegurar tipo numérico (por las dudas)
cols_edu = ["Nivel_Ed_Madre", "Nivel_Ed_Padre"]
df[cols_edu] = df[cols_edu].apply(pd.to_numeric, errors="coerce")

# Unir al df1
df1 = df1.merge(df[["idalumno"] + cols_edu], on="idalumno", how="left")

In [125]:
# Aseguramos que sea numérica por las dudas
df["clima_escolar"] = pd.to_numeric(df["clima_escolar"], errors="coerce")

# Unimos a df1
df1 = df1.merge(df[["idalumno", "clima_escolar"]], on="idalumno", how="left")

In [126]:
df1.isna().sum()

idalumno               0
ap01_edad_index        0
sector_privado         0
ambito_urbano          0
sexo_2.0               0
                   ...  
migracion          38230
sobreedad          19514
Nivel_Ed_Madre     19514
Nivel_Ed_Padre     19514
clima_escolar      20543
Length: 62, dtype: int64

In [127]:
df1=df1.dropna()

In [86]:
df1.isna().sum()

idalumno           0
ap01_edad_index    0
sector_privado     0
ambito_urbano      0
sexo_2.0           0
                  ..
migracion          0
sobreedad          0
Nivel_Ed_Madre     0
Nivel_Ed_Padre     0
clima_escolar      0
Length: 75, dtype: int64

PCA

In [130]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1️⃣ Definir las variables que vas a incluir en el PCA (todas menos idalumno)
X = df1.drop(columns=["idalumno"])

# 2️⃣ Escalar los datos (media=0, varianza=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3️⃣ Aplicar PCA
pca = PCA(n_components=0.9)  # Retiene componentes que explican el 90% de la varianza total
#pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 4️⃣ Ver cuánta varianza explica cada componente
import pandas as pd
explained_var = pd.DataFrame({
    "Componente": range(1, len(pca.explained_variance_ratio_) + 1),
    "Varianza explicada (%)": pca.explained_variance_ratio_ * 100,
    "Varianza acumulada (%)": pca.explained_variance_ratio_.cumsum() * 100
})
explained_var.head(10)

,Componente,Varianza explicada (%),Varianza acumulada (%)
0,1,10.416608,10.416608
1,2,5.479134,15.895742
2,3,5.202786,21.098527
3,4,4.667605,25.766132
4,5,3.871034,29.637166
5,6,2.914969,32.552135
6,7,2.615573,35.167708
7,8,2.312829,37.480537
8,9,2.200312,39.680849
9,10,2.050123,41.730972


In [131]:
# Crear un DataFrame con los loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    index=X.columns
)

# Mostrar las 10 variables que más contribuyen al primer componente
print("🔹 Variables más importantes en el PC1:")
print(loadings["PC1"].abs().sort_values(ascending=False).head(10))

🔹 Variables más importantes en el PC1:
ap19a      0.322519
ap19c      0.320956
ap19d      0.319199
ap19b      0.317896
ap18a      0.315577
ap18c      0.313936
ap18b      0.312233
ap18d      0.310774
ldesemp    0.163674
mdesemp    0.145073
Name: PC1, dtype: float64


In [93]:
df1.head()

,idalumno,ap01_edad_index,sector_privado,ambito_urbano,sexo_2.0,sexo_3.0,ap05_index,ap06_index,ap07_index,ap09_index,...,apA36_index_negativo_y,indice_discriminacion,ldesemp,mdesemp,NSE_nivel,migracion,sobreedad,Nivel_Ed_Madre,Nivel_Ed_Padre,clima_escolar
0,2,2.0,0,1,0,0,0.6875,1.875,1.0,0.863636,...,10.0,0.0,1.0,2.0,2.0,0.0,0.0,5.0,3.0,2.0
1,2,2.0,0,1,0,0,0.6875,1.875,1.0,0.863636,...,10.0,0.0,1.0,2.0,2.0,0.0,0.0,5.0,3.0,2.0
2,2,2.0,0,1,0,0,0.6875,1.875,1.0,0.863636,...,10.0,0.0,1.0,2.0,2.0,0.0,0.0,6.0,6.0,2.0
3,2,2.0,0,1,0,0,0.6875,1.875,1.0,0.863636,...,10.0,0.0,1.0,2.0,2.0,0.0,0.0,6.0,6.0,2.0
4,2,2.0,0,1,0,0,0.6875,1.875,1.0,0.863636,...,10.0,0.0,1.0,2.0,2.0,0.0,0.0,5.0,3.0,2.0


In [128]:
df1.columns.tolist()

['idalumno',
 'ap01_edad_index',
 'sector_privado',
 'ambito_urbano',
 'sexo_2.0',
 'sexo_3.0',
 'ap05_index',
 'ap06_index',
 'ap07_index',
 'ap09_index',
 'ap10_libros',
 'ap11_si_no',
 'ap12_si_no',
 'ap13_si_no',
 'ap14_cantidad',
 'ap15_cantidad',
 'ap18a',
 'ap18b',
 'ap18c',
 'ap18d',
 'ap19a',
 'ap19b',
 'ap19c',
 'ap19d',
 'ap22_1.0',
 'ap22_2.0',
 'ap22_3.0',
 'ap22_4.0',
 'ap24a',
 'ap24b',
 'ap24c',
 'ap24d',
 'ap24e',
 'ap24f',
 'ap24g',
 'ap24h',
 'ap24i',
 'ap24j',
 'ap24k',
 'ap25a',
 'ap25b',
 'ap25c',
 'ap25d',
 'ap25e',
 'ap25f',
 'ap25g',
 'ap25h',
 'ap25i',
 'ap25j',
 'ap26',
 'indice_apA34',
 'apA36_positivo',
 'apA36_index_negativo_x',
 'apA36_index_negativo_y',
 'ldesemp',
 'mdesemp',
 'NSE_nivel',
 'migracion',
 'sobreedad',
 'Nivel_Ed_Madre',
 'Nivel_Ed_Padre',
 'clima_escolar']

In [129]:
df1.shape

(51553, 62)

In [132]:
df["ap23"] = pd.to_numeric(df["ap23"], errors="coerce").replace({-9: np.nan, -8: np.nan, -6: np.nan})
df = df.dropna(subset=["ap23"])
df["ap23"].value_counts(dropna=False)

ap23
2.0    209453
3.0    106447
4.0     77734
1.0     62644
6.0     45085
5.0     38788
Name: count, dtype: int64

In [133]:
# Crear una versión categórica (bajo / medio / alto ausentismo)
def categorizar_faltas(x):
    if x in [1, 2]:
        return "bajo"
    elif x in [3, 4]:
        return "medio"
    elif x in [5, 6]:
        return "alto"
    else:
        return np.nan  # elimina los -9, -8, etc.

df["ausentismo_cat"] = df["ap23"].apply(categorizar_faltas)

In [134]:
# Unir df1 con la variable a predecir
data = df1.merge(df[["idalumno", "ausentismo_cat"]], on="idalumno", how="inner")

# Separar variables predictoras y target
X = data.drop(columns=["idalumno", "ausentismo_cat"])
y = data["ausentismo_cat"]

In [135]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [136]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

In [141]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=29)  # podés ajustar este valor después
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=29)

In [142]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = knn.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[  95 1759  391]
 [  45 7414  642]
 [  52 3568  722]]
              precision    recall  f1-score   support

        alto       0.49      0.04      0.08      2245
        bajo       0.58      0.92      0.71      8101
       medio       0.41      0.17      0.24      4342

    accuracy                           0.56     14688
   macro avg       0.50      0.37      0.34     14688
weighted avg       0.52      0.56      0.47     14688



In [139]:
y.value_counts(normalize=True)


ausentismo_cat
bajo     0.551534
medio    0.295621
alto     0.152845
Name: proportion, dtype: float64

In [140]:
from sklearn.model_selection import cross_val_score
import numpy as np

ks = range(1, 31)
scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_scaled, y, cv=5).mean() for k in ks]

best_k = ks[np.argmax(scores)]
print(f"Mejor k: {best_k}")

Mejor k: 29
